In [ ]:
import pandas as pd
import sqlite3

# Load dataset
df = pd.read_csv('shop_dataset.csv')

# Connect to SQLite in-memory database
conn = sqlite3.connect(':memory:')
df.to_sql('pos_transactions', conn, index=False)

# Intermediate SQL Query: Payment Method Breakdown
payment_query = """
    SELECT 
        Transaction_Type,
        COUNT(CustomerID) AS Total_Transactions,
        SUM(Price) AS Total_Revenue,
        ROUND(AVG(Price), 2) AS Average_Transaction_Value
    FROM pos_transactions
    GROUP BY Transaction_Type
    ORDER BY Total_Revenue DESC;
"""
print("--- Payment Method Insights ---")
print(pd.read_sql_query(payment_query, conn))

In [ ]:
monthly_trend_query = """
    SELECT 
        SUBSTR(Date, 4, 2) AS Month, -- extracting MM from DD-MM-YYYY format
        COUNT(CustomerID) AS Monthly_Orders,
        SUM(Price) AS Monthly_Revenue
    FROM pos_transactions
    GROUP BY Month
    ORDER BY Month ASC;
"""
print("\n--- Monthly Revenue Trends ---")
print(pd.read_sql_query(monthly_trend_query, conn))

In [ ]:
demographic_query = """
    WITH CustomerSpending AS (
        SELECT 
            CustomerID,
            Name,
            Sex,
            Age,
            SUM(Price) AS Total_Spent,
            COUNT(Product_ID) AS Items_Bought
        FROM pos_transactions
        GROUP BY CustomerID, Name, Sex, Age
    ),
    RankedCustomers AS (
        SELECT 
            Name,
            Sex,
            Age,
            Total_Spent,
            DENSE_RANK() OVER (ORDER BY Total_Spent DESC) as Spending_Rank
        FROM CustomerSpending
    )
    SELECT * FROM RankedCustomers
    WHERE Spending_Rank <= 5;
"""
print("\n--- Top High-Value Customers (Window Function & CTE) ---")
print(pd.read_sql_query(demographic_query, conn))